In [3]:
import pandas as pd

df = pd.read_csv(
    "../Data/raw/olist_orders_dataset.csv",
    nrows=0
)

print(df.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [4]:
df = pd.read_csv(
    "../Data/raw/olist_orders_dataset.csv",
    usecols=["order_estimated_delivery_date"]
)

print("Missing values:", df["order_estimated_delivery_date"].isna().sum())

Missing values: 0


In [6]:
%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pyodbc

print(pyodbc.drivers())


['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [ ]:
import pyodbc
# Update SERVER with your own SQL Server instance name.
# Example: YOUR_COMPUTER_NAME\SQLEXPRESS

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-NCV6NCI\\SQLEXPRESS;"
    "DATABASE=SupplyChain_Source;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(connection_string)

print("Connected successfully!")

Connected successfully!


In [12]:
import pandas as pd

orders_df = pd.read_csv(
    "../Data/raw/olist_orders_dataset.csv"
)

print(orders_df.shape)
print(orders_df.dtypes)
print(orders_df.isna().sum())

(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [13]:
import pandas as pd

# Convert date columns
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_df[col] = pd.to_datetime(orders_df[col], errors="coerce")

# Replace NaN / NaT with None
orders_df = orders_df.astype(object).where(pd.notna(orders_df), None)

print(orders_df.dtypes)
print("Rows ready:", len(orders_df))

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
Rows ready: 99441


In [14]:
cursor = conn.cursor()
cursor.fast_executemany = True

insert_query = """
INSERT INTO source.orders (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

data = list(orders_df.itertuples(index=False, name=None))

try:
    cursor.executemany(insert_query, data)
    conn.commit()
    print(f"Successfully loaded {cursor.rowcount} orders.")

except Exception as e:
    conn.rollback()
    print("Loading failed. Transaction rolled back.")
    print(e)

finally:
    cursor.close()

Successfully loaded -1 orders.
